# D4b 트랜스포머와 사전학습 — 실습 (W11, D4 2부작 완결편)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> (HuggingFace 모델은 첫 실행 시 자동 다운로드됩니다 — 약 250MB×2. 이 수업 PC에는 캐시돼 있습니다.)

**이 실습이 끝나면**
1. 위치 인코딩 함수를 완성하고 **손계산 표**(pos=1 행 [0.84, 0.54, 0.01, 1.00])를 검증한다
2. **순서 실험 2종** — PE가 없으면 순서 무지(allclose=True), 더하면 깨짐(False)
3. 트랜스포머 블록의 **가격표 33,472개**를 numel로 검산한다
4. **뉴스 분류기 3라운드** — Embedding+PE+블록으로 **85.5%** (+ 정직한 해부)
5. **HuggingFace pipeline**으로 사전학습 모델(감정분석·fill-mask)을 가져다 쓴다

**7단계 멘탈모델 초점:** 모델(블록 조립) + 활용(사전학습 모델)

## Part A. 위치 인코딩 — 손계산 검증 ⭐
짝수 차원 = sin, 홀수 차원 = cos. d=4면 분모는 1과 100 두 개뿐.
**먼저 종이에서 pos=1 행을 완주한 뒤** 실행해 답을 맞춰 보세요.

In [ ]:
import math                                          # 수학 함수(검산용)
import torch                                          # PyTorch
import torch.nn as nn                                 # 신경망 모듈
import matplotlib.pyplot as plt                       # 그래프

def positional_encoding(max_len, d):                  # sin/cos 위치 인코딩 표
    pos = torch.arange(max_len).unsqueeze(1).float()  # (max_len, 1) 위치 번호
    i = torch.arange(0, d, 2).float()                 # 짝수 차원 번호 0, 2, ...
    div = 10000 ** (i / d)                            # 분모(파장) — d=4면 [1, 100]
    pe = torch.zeros(max_len, d)                      # 표 준비
    pe[:, 0::2] = torch.sin(pos / div)                # 짝수 차원 = sin
    pe[:, 1::2] = torch.___(pos / div)                # ✍️ 빈칸: 홀수 차원은 어떤 함수?
    return pe

pe4 = positional_encoding(4, 4)                       # 위치 4개 × 차원 4
for p in range(4):                                    # 본문 §4의 표와 대조
    print(f'pos {p}:', [round(v, 2) for v in pe4[p].tolist()])
print('검산: sin(1) =', round(math.sin(1), 2), '| cos(1) =', round(math.cos(1), 2))

> **검산 포인트:** pos=1 행 = [0.84, 0.54, 0.01, 1.00] — 손계산과 일치?
> 앞 두 차원은 **빠른 시계**(1라디안씩), 뒤 두 차원은 **100배 느린 시계** — 여러 속도의 조합이 자릿수 역할.

In [ ]:
pe = positional_encoding(100, 64)                     # 위치 100 × 차원 64로 확장
fig, ax = plt.subplots(figsize=(7, 3.2))
im = ax.imshow(pe.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)  # 행=차원, 열=위치
ax.set_xlabel('position'); ax.set_ylabel('dimension')  # 라벨은 영어(Colab 한글 폰트)
ax.set_title('Sinusoidal positional encoding (d=64)')
fig.colorbar(im, fraction=0.03, pad=0.02); fig.tight_layout(); plt.show()

> 위쪽(앞 차원)은 줄무늬가 촘촘(빠른 파형), 아래로 갈수록 느슨(느린 파형). 어떤 위치를 세로로 잘라도 **서로 다른 무늬** — 모든 위치가 고유한 벡터를 받습니다. 값은 항상 −1~1(임베딩을 압도하지 않음).

## Part B. 순서 실험 2종 — PE가 심는 것
**실험 ①** 같은 단어, 다른 자리. **실험 ②** 어텐션+평균은 순서를 잊는다 → PE가 깨운다.

In [ ]:
torch.manual_seed(0)                                  # 재현성
e_dog = torch.randn(4)                                # 단어 "개"의 임베딩(가상, d=4)
v1 = e_dog + ___[1]                                   # ✍️ 빈칸: 1번 자리의 개 = 임베딩 + 위치 벡터 표의 1번 행
v3 = e_dog + pe4[3]                                   # 3번 자리의 개
print('1번 자리의 개:', [round(v, 2) for v in v1.tolist()])
print('3번 자리의 개:', [round(v, 2) for v in v3.tolist()])
print('두 벡터가 같은가?', torch.allclose(v1, v3))    # False — 자리가 다르면 다른 입력!

In [ ]:
def attention(Q, K, V):                               # D4a의 완성형 그대로
    d_k = Q.size(-1)                                  # Key 차원
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)   # 점수 ÷ √dₖ
    attn = torch.softmax(scores, dim=-1)              # 행(Query)별 가중치
    return attn @ V, attn                             # Value 가중합

torch.manual_seed(0)                                  # 재현성
E = torch.randn(4, 8)                                 # 4단어 문장(가상 임베딩, d=8)
E_swap = E[torch.tensor([3, 1, 2, 0])]                # 첫 단어 ↔ 끝 단어를 맞바꾼 문장

o1, _ = attention(E, E, E)                            # 원문 통과
o2, _ = attention(E_swap, E_swap, E_swap)             # 맞바꾼 문장 통과
print('PE 없이 문서벡터 같은가?', torch.allclose(o1.mean(0), o2.mean(0), atol=1e-6))

pe8 = positional_encoding(4, 8)                       # 위치 벡터(d=8)
o1p, _ = attention(E + pe8, E + pe8, E + pe8)         # 위치를 더한 원문
o2p, _ = attention(E_swap + pe8, E_swap + pe8, E_swap + pe8)  # 위치를 더한 맞바꾼 문장
print('PE 더함 문서벡터 같은가?', torch.allclose(o1p.mean(0), o2p.mean(0), atol=1e-6))
print('최대 차이:', round((o1p.mean(0) - o2p.mean(0)).abs().max().item(), 4))

> **PE 없이 True**(개가 사람을 = 사람이 개를 — 순서 무지), **PE 더하면 False**(최대 차이 0.2173). 순서가 입력에 심어졌습니다. 단, 이것은 "순서를 **쓸 수 있는 능력**"이지 성능 부스터가 아님 — Part D에서 정직하게 해부합니다.

## Part C. 트랜스포머 블록 — 가격표 검산
블록 = 멀티헤드 어텐션(섞기) + FFN(다듬기) + Add&Norm(잔차연결 — D2b ResNet의 재회) ×2.
D1c의 numel로 **손계산 33,472**를 검산합니다.

In [ ]:
torch.manual_seed(0)                                  # 재현성
block = nn.TransformerEncoderLayer(d_model=64, nhead=___,  # ✍️ 빈칸: 64를 나누어떨어뜨리는 head 수(헤드당 16차원)
                                   dim_feedforward=128, batch_first=True)  # FFN 폭 128, (배치,시간,특징)
n_block = sum(p.numel() for p in block.parameters())  # 파라미터 세기(D1c numel)
print('블록 파라미터:', f'{n_block:,}')               # 손계산 33,472와 일치?
attn_p = sum(p.numel() for n, p in block.named_parameters() if 'self_attn' in n)  # 어텐션 몫
ffn_p = sum(p.numel() for n, p in block.named_parameters() if 'linear' in n)      # FFN 몫
ln_p = sum(p.numel() for n, p in block.named_parameters() if 'norm' in n)         # LayerNorm 몫
print('어텐션:', f'{attn_p:,}', '| FFN:', f'{ffn_p:,}', '| LayerNorm:', ln_p)

x = torch.randn(1, 5, 64)                             # (배치 1, 단어 5, 차원 64)
print('블록 통과 후 shape:', tuple(block(x).shape))   # 모양 보존 — 그래서 N층으로 쌓을 수 있다

> 어텐션 16,640 + FFN 16,576 + LayerNorm 256 = **33,472** — 본문 §7 가격표와 일치. 입력과 출력의 모양이 같으므로 블록을 계속 쌓을 수 있습니다(위치 인코딩은 파라미터 **0개**).

## Part D. 실전 3라운드 — 뉴스 분류기에 블록을 얹다 ⭐
D3b·D4a와 **완전히 같은 데이터·전처리**(야구 vs 우주). Embedding+PE → 블록 1층 → 평균 풀링.

In [ ]:
import re                                             # 토큰화(D3b 그대로)
from sklearn.datasets import fetch_20newsgroups       # 뉴스 데이터
from collections import Counter                       # 단어 빈도

def tok(t):                                           # D3b의 토크나이저 그대로
    return re.findall(r'[a-z]+', t.lower())

cats = ['rec.sport.baseball', 'sci.space']            # 두 주제
tr = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'))
te = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'))

counter = Counter(w for d in tr.data for w in tok(d)) # 빈도는 train으로만(M2 누수 방지)
itos = ['<pad>', '<unk>'] + [w for w, _ in counter.most_common(5000)]  # 예약 2 + 상위 5000
stoi = {w: i for i, w in enumerate(itos)}             # 단어→정수

MAX = 200                                             # 고정 길이(앞쪽 패딩)
def encode(t):                                        # 문서 → 길이 200 정수 시퀀스
    ids = [stoi.get(w, 1) for w in tok(t)][:MAX]      # 인코딩(+길면 자르기)
    return [0] * (MAX - len(ids)) + ids               # 앞쪽을 <pad>로

Xtr = torch.tensor([encode(d) for d in tr.data]); ytr = torch.tensor(tr.target)
Xte = torch.tensor([encode(d) for d in te.data]); yte = torch.tensor(te.target)
print('train:', tuple(Xtr.shape), '| test:', tuple(Xte.shape))  # (1190, 200) / (791, 200)

In [ ]:
torch.manual_seed(0)                                  # 재현성
class TransformerClassifier(nn.Module):               # 임베딩+PE → 블록 → 평균 → Linear
    def __init__(self, vocab_size, embed=64, nclass=2, max_len=200):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed, padding_idx=0)  # D3b 그대로
        self.register_buffer('pe', positional_encoding(max_len, embed))  # 위치 표(파라미터 0개)
        self.block = nn.TransformerEncoderLayer(d_model=embed, nhead=4,
                                                dim_feedforward=128, batch_first=True)  # 블록 1층
        self.fc = nn.Linear(embed, nclass)            # 문서 벡터 → 클래스 점수
    def forward(self, x):
        e = self.embedding(x) + self.pe               # 임베딩 + 위치 (concat 아닌 더하기!)
        pad = (x == ___)                              # ✍️ 빈칸: <pad>의 번호 — D4a 마스킹의 그 값
        pad[:, -1] = False                            # 빈 문서 안전장치(전부 pad면 어텐션이 NaN)
        h = self.block(e, src_key_padding_mask=pad)   # D4a의 −10⁹ 마스킹을 내장 지원으로
        m = (~pad).float().unsqueeze(-1)              # 실제 단어 위치만 1
        doc = (h * m).sum(dim=___) / m.sum(dim=1).clamp(min=1)  # ✍️ 빈칸: 시간(단어) 차원 평균
        return self.fc(doc)                           # (B, 2)

model = TransformerClassifier(len(itos))              # 생성
n_total = sum(p.numel() for p in model.parameters())  # 전체
n_emb = sum(p.numel() for p in model.embedding.parameters())  # 임베딩 몫
n_blk = sum(p.numel() for p in model.block.parameters())      # 블록 몫
print('전체:', f'{n_total:,}', '= 임베딩', f'{n_emb:,}', '+ 블록', f'{n_blk:,}', '+ fc 130')
print('참고: D3b LSTM은 353,538 — 거의 같은 예산, 구조만 교체!')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader  # 배치 공급

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()                     # 분류 손실(D1b)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam

losses = []                                           # epoch 손실 기록
for epoch in range(5):                                # D3b·D4a와 동일 조건(5바퀴)
    model.train(); running = 0.0
    for xb, yb in train_loader:                       # D1c 5단계 그대로
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    losses.append(running / len(train_loader))
    print(f'epoch {epoch+1}: train loss = {losses[-1]:.4f}')

model.eval()                                          # 평가 스위치(D1c)
with torch.no_grad():
    acc = (model(Xte).argmax(-1) == yte).float().mean().item()
print('test accuracy:', round(acc, 3))                # ~0.855
print('스코어보드: D3b 0.741 → D4a 0.823 → D4b', round(acc, 3))  # 한 계단 더

plt.figure(figsize=(6, 3.5))                          # 손실 곡선
plt.plot(range(1, 6), losses, 'o-')                   # epoch별
plt.xlabel('epoch'); plt.ylabel('train loss')         # 축(영어)
plt.xticks(range(1, 6))                               # 정수 눈금
plt.title(f'Transformer block classifier (test acc {acc:.3f})')  # 제목(영어)
plt.grid(True); plt.show()

> **85.5%** — 74.1(D3b) → 82.3(D4a) → 85.5. D3b와 거의 같은 파라미터 예산(353,730 vs 353,538)으로 **+11.4%p** — "구조가 용량을 이긴다"(D2b)를 같은 용량에서 재확인.

In [ ]:
torch.manual_seed(1)                                  # 재현성(섞기용)
idx = torch.randperm(MAX)                             # 200개 자리를 무작위로 뒤섞기
with torch.no_grad():
    acc_shuf = (model(Xte[:, idx]).argmax(-1) == yte).float().mean().item()
print('단어를 뒤섞은 입력의 정확도:', round(acc_shuf, 3))  # ~0.861 — 거의 그대로!

> **정직한 해부:** 뒤섞어도 0.861 — 이 과제에서 **순서는 승부처가 아니었습니다**(주제 분류 = "어떤 단어가 있나"의 게임). +3.2%p의 공은 **블록**(멀티헤드로 섞고 FFN으로 다듬은 깊은 상호작용). PE의 진짜 무대는 순서가 의미를 바꾸는 번역·생성 — 아래 사전학습 모델의 세계입니다.

## Part E. HuggingFace — 사전학습 모델 가져다 쓰기
DistilBERT(6,700만 개 — 우리 블록의 약 2,000배)를 **몇 줄로**. 사전학습+미세조정 = D2c 전이학습의 NLP판.

In [ ]:
from transformers import pipeline                     # HuggingFace(Colab: pip install transformers)

clf = pipeline('___',                                 # ✍️ 빈칸: 감정분석 작업 이름
               model='distilbert-base-uncased-finetuned-sst-2-english')  # 사전학습+SST-2 미세조정판
for s in ['This movie was fantastic!', 'The food was terrible and cold.']:
    r = clf(s)[0]                                     # 토큰화→추론→후처리 자동
    print(s, '→', r['label'], round(r['score'], 3))

In [ ]:
unmasker = pipeline('fill-mask', model='distilbert-base-uncased')  # 사전학습 과제 그 자체
mt = unmasker.tokenizer.___                           # ✍️ 빈칸: 이 토크나이저의 마스크 토큰 속성 이름
print('마스크 토큰:', mt)                             # [MASK]
for s in [f'Paris is the {mt} of France.',            # 상식형 빈칸
          f'I went to the bank to deposit some {mt}.']:  # 문맥형 빈칸(bank=은행? 강둑?)
    print(s)
    for r in unmasker(s)[:3]:                         # 상위 3개 후보
        print('   ', r['token_str'].strip(), round(r['score'], 2))

In [ ]:
n_bert = sum(p.numel() for p in unmasker.model.parameters())  # DistilBERT 크기(D1c numel)
print('DistilBERT 파라미터:', f'{n_bert:,}')          # 66,985,530
print('우리 블록(33,472)의 약', round(n_bert / n_block), '배')  # ~2,000배

> capital(0.98) — 상식. money(0.42)·cash(0.21) — **deposit 문맥**이 bank를 은행으로 읽게 했습니다(D4a "문맥이 반영된 표현"의 실전). 빈칸 맞히기가 바로 **사전학습 과제(Masked LM)** — 원문이 정답이라 라벨 비용 0원, 그래서 수십 GB로 키울 수 있었습니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "pos=1 행 [0.84, 0.54, 0.01, 1.00]을 공식에서 유도해 볼 테니 채점해 줘."
- "왜 concat이 아니라 더하기인지 두 가지 이유(차원·병렬성)로 설명해 볼게."
- "33,472를 부품별로 나눠 계산했어 — 각 항이 맞는지 확인해 줘."
- "뒤섞어도 0.861인데 PE가 왜 필요한지 내 말로 설명해 볼게 — 반례가 되는 과제를 들어 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 위치 인코딩을 완성해 손계산 표를 검증하고, 순서 실험 2종으로 "PE가 심는 것"을 확인했다
2. 블록 가격표 33,472를 검산하고, Embedding+PE+블록으로 **85.5%**(같은 예산, +11.4%p)를 만들었다
3. 뒤섞기 재실험(0.861)으로 상승의 공을 정직하게 해부하고, HuggingFace로 사전학습 모델을 체험했다

**스스로 점검**
- [ ] "더하기(concat 아님)"의 이유 두 가지를 말할 수 있다
- [ ] `src_key_padding_mask`가 D4a의 −10⁹ 마스킹과 같은 원리임을 안다
- [ ] PE가 성능 부스터가 아니라 "순서를 쓸 능력"인 이유를 실측으로 설명할 수 있다
- [ ] BERT형(빈칸)과 GPT형(다음 단어)의 사전학습 과제 차이를 안다

**🔹심화 (선택)**
- **블록 2층으로:** `nn.TransformerEncoder(block, num_layers=2)`로 쌓으면 성능이 오를까요? 파라미터와 과적합(M2)을 함께 관찰하세요.
- **PE 빼고 재학습:** `+ self.pe`를 지우고 다시 학습 — 이 과제에서 정확도가 얼마나 변하는지 실측(예상: 거의 그대로 — 왜?).
- **서브워드 구경:** `unmasker.tokenizer.tokenize('huggingface')` → `['hugging', '##face']` — 단어 조각 사전(3만 개)이라 <unk>가 없습니다. D3b의 5,000 단어 사전과 비교해 보세요.
- **다른 pipeline:** `pipeline('summarization')`, `pipeline('ner')` 등 — 모두 같은 트랜스포머 골격입니다.